In [24]:
import polars as pl

In [25]:
f = open('movies.dat', 'r', encoding="ISO-8859-1")
f = open('users.dat', 'r')
f = open('ratings.dat', 'r')

In [26]:
movies = pl.read_csv(
    "movies.dat",
    has_header=False,
    separator="\n",              # ✅ 每行当成一个字段
    new_columns=["raw"],
    encoding="latin1",
)
movies = (
    movies
    .with_columns(pl.col("raw").str.split("::"))
    .with_columns([
        pl.col("raw").list.get(0).cast(pl.Int32).alias("movieId"),
        pl.col("raw").list.get(1).alias("title"),
        pl.col("raw").list.get(2).alias("genres"),
    ])
    .select(["movieId", "title", "genres"])
)

In [27]:
movies.head()

movieId,title,genres
i32,str,str
1,"""Toy Story (1995)""","""Animation|Children's|Comedy"""
2,"""Jumanji (1995)""","""Adventure|Children's|Fantasy"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama"""
5,"""Father of the Bride Part II (1…","""Comedy"""


In [34]:
movies_feat = (
    movies
    .with_columns([
        # 提取年份
        pl.col("title").str.extract(r"\((\d{4})\)").cast(pl.Int32).alias("year"),

        # 拆分 genre
        pl.col("genres").str.split("|").alias("genre_list"),
    ])
    .select(["movieId", "year", "genre_list"])
)
genre_list = (
    movies_feat
    .select("genre_list")
    .explode("genre_list")
    .unique()
    .to_series()
    .to_list()
)
genre2idx = {g: i + 1 for i, g in enumerate(genre_list)}

In [29]:
users = pl.read_csv("users.dat", separator="\n", has_header=False, new_columns=["raw"])

users = users.with_columns(pl.col("raw").str.split("::")).with_columns([
    pl.col("raw").list.get(0).cast(pl.Int32).alias("userId"),
    pl.col("raw").list.get(1).alias("gender"),
    pl.col("raw").list.get(2).alias("age"),
    pl.col("raw").list.get(3).alias("occupation"),
    pl.col("raw").list.get(4).alias("zipcode"),
]).select(["userId", "gender", "age", "occupation", "zipcode"])

In [30]:
users.head()

userId,gender,age,occupation,zipcode
i32,str,str,str,str
1,"""F""","""1""","""10""","""48067"""
2,"""M""","""56""","""16""","""70072"""
3,"""M""","""25""","""15""","""55117"""
4,"""M""","""45""","""7""","""02460"""
5,"""M""","""25""","""20""","""55455"""


In [31]:
users['age'].value_counts()

age,count
str,u32
"""25""",2096
"""56""",380
"""1""",222
"""50""",496
"""18""",1103
"""45""",550
"""35""",1193


In [32]:
ratings = (
    pl.read_csv(
        "ratings.dat",
        has_header=False,
        separator="\n",     # 每行作为一个字段
        new_columns=["raw"],
    )
    .with_columns(pl.col("raw").str.split("::"))
    .with_columns([
        pl.col("raw").list.get(0).cast(pl.Int32).alias("userId"),
        pl.col("raw").list.get(1).cast(pl.Int32).alias("movieId"),
        pl.col("raw").list.get(2).cast(pl.Int32).alias("rating"),
        pl.col("raw").list.get(3).cast(pl.Int64).alias("timestamp"),
    ])
    .select(["userId", "movieId", "rating", "timestamp"])
)

In [33]:
ratings.head()

userId,movieId,rating,timestamp
i32,i32,i32,i64
1,1193,5,978300760
1,661,3,978302109
1,914,3,978301968
1,3408,4,978300275
1,2355,5,978824291
